# 치매알람 성능 평가 통합 노트북

- 섹션 1: GraphDB 검색 성능 (Precision / Recall)
- 섹션 2: VectorDB 검색 성능 (Hit Rate / MRR)
- 섹션 3: RAG 모델 성능 (RAGAS)

> 실행 전 프로젝트 루트에서 실행하거나 sys.path에 루트를 추가해야 합니다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # 프로젝트 루트 기준

from dotenv import load_dotenv
load_dotenv()
print("환경변수 로드 완료")

환경변수 로드 완료


---
## 1. GraphDB 성능 평가 (Precision / Recall)

9개 고정 tool + `flexible_graph_search` 결과를 Ground Truth와 비교한다.  
Ground Truth는 직접 작성한 정답 센터명 목록이다.

In [2]:
from graph_db.graph_search_tool import (
    get_centers_by_sido,
    get_centers_by_sigungu,
    get_centers_by_program,
    get_programs_by_center,
    search_center_by_name,
    get_operator_by_center,
    get_centers_by_operator,
    get_sido_list,
    get_sigungu_list,
    flexible_graph_search,
)
print("GraphDB tool import 완료")

GraphDB tool import 완료


In [7]:
# 평가셋: (설명, tool_func, tool_args, ground_truth_keywords)
# ground_truth_keywords: 결과 텍스트에 반드시 포함되어야 할 키워드 목록
GRAPH_EVAL_SET = [
    {
        "desc": "서울특별시 치매안심센터 조회",
        "tool": get_centers_by_sido,
        "args": {"sido": "서울특별시"},
        "keywords": ["강남구", "서초구", "종로구"],
    },
    {
        "desc": "강남구 치매안심센터 조회",
        "tool": get_centers_by_sigungu,
        "args": {"sido": "서울특별시", "sigungu": "강남구"},
        "keywords": ["강남구"],
    },
    {
        "desc": "인지훈련 프로그램 센터 검색",
        "tool": get_centers_by_program,
        "args": {"program_keyword": "인지훈련"},
        "keywords": ["인지훈련"],
    },
    {
        "desc": "삼성서울병원 운영 센터 조회",
        "tool": get_centers_by_operator,
        "args": {"operator_name": "삼성서울병원"},
        "keywords": ["강남구치매안심센터"],  # ← 변경
    },
    {
        "desc": "센터명 키워드 검색 - 강남",
        "tool": search_center_by_name,
        "args": {"keyword": "강남"},
        "keywords": ["강남"],
    },
    {
        "desc": "서울 강남구 치매안심센터 운영기관 조회",
        "tool": get_operator_by_center,
        "args": {"center_name": "서울특별시강남구치매안심센터"},
        "keywords": ["삼성서울병원"],  # ← 변경
    },
    {
        "desc": "센터 제공 프로그램 조회",
        "tool": get_programs_by_center,
        "args": {"center_name": "서울특별시강남구치매안심센터"},
        "keywords": ["프로그램"],
    },
    {
        "desc": "시도 목록 조회",
        "tool": get_sido_list,
        "args": {},
        "keywords": ["서울특별시", "경기도", "부산광역시"],
    },
    {
        "desc": "시군구 목록 조회 - 경기도",
        "tool": get_sigungu_list,
        "args": {"sido": "경기도"},
        "keywords": ["수원시", "성남시"],
    },
    {
        "desc": "flexible - 의사 2명 이상 센터 (fallback)",
        "tool": flexible_graph_search,
        "args": {"query": "의사가 2명 이상인 치매안심센터는 어디인가요?"},
        "keywords": ["센터"],
    },
]

In [8]:
def eval_graph_tool(tool_func, args, keywords):
    """tool 호출 결과에 keyword가 포함되면 hit"""
    try:
        result = tool_func.invoke(args)
        hits = [kw for kw in keywords if kw in result]
        precision = len(hits) / len(keywords) if keywords else 0
        recall = len(hits) / len(keywords) if keywords else 0
        return result, precision, recall, hits
    except Exception as e:
        return str(e), 0, 0, []


graph_results = []
for item in GRAPH_EVAL_SET:
    result, precision, recall, hits = eval_graph_tool(
        item["tool"], item["args"], item["keywords"]
    )
    status = "✅" if precision == 1.0 else ("⚠️" if precision > 0 else "❌")
    graph_results.append({
        "desc": item["desc"],
        "tool": item["tool"].name,
        "precision": precision,
        "recall": recall,
        "status": status,
        "result_preview": result[:80] if isinstance(result, str) else str(result)[:80],
    })
    print(f"{status} [{item['tool'].name}] {item['desc']} | P={precision:.2f} R={recall:.2f}")

avg_p = sum(r["precision"] for r in graph_results) / len(graph_results)
avg_r = sum(r["recall"] for r in graph_results) / len(graph_results)
print(f"\n평균 Precision: {avg_p:.4f}")
print(f"평균 Recall:    {avg_r:.4f}")

✅ [get_centers_by_sido] 서울특별시 치매안심센터 조회 | P=1.00 R=1.00
✅ [get_centers_by_sigungu] 강남구 치매안심센터 조회 | P=1.00 R=1.00
✅ [get_centers_by_program] 인지훈련 프로그램 센터 검색 | P=1.00 R=1.00
✅ [get_centers_by_operator] 삼성서울병원 운영 센터 조회 | P=1.00 R=1.00
✅ [search_center_by_name] 센터명 키워드 검색 - 강남 | P=1.00 R=1.00
✅ [get_operator_by_center] 서울 강남구 치매안심센터 운영기관 조회 | P=1.00 R=1.00
✅ [get_programs_by_center] 센터 제공 프로그램 조회 | P=1.00 R=1.00
✅ [get_sido_list] 시도 목록 조회 | P=1.00 R=1.00
✅ [get_sigungu_list] 시군구 목록 조회 - 경기도 | P=1.00 R=1.00
✅ [flexible_graph_search] flexible - 의사 2명 이상 센터 (fallback) | P=1.00 R=1.00

평균 Precision: 1.0000
평균 Recall:    1.0000


In [9]:
import pandas as pd
df_graph = pd.DataFrame(graph_results)[["status", "desc", "tool", "precision", "recall"]]
df_graph.columns = ["상태", "질문", "사용 tool", "Precision", "Recall"]
df_graph

,상태,질문,사용 tool,Precision,Recall
0,✅,서울특별시 치매안심센터 조회,get_centers_by_sido,1.0,1.0
1,✅,강남구 치매안심센터 조회,get_centers_by_sigungu,1.0,1.0
2,✅,인지훈련 프로그램 센터 검색,get_centers_by_program,1.0,1.0
3,✅,삼성서울병원 운영 센터 조회,get_centers_by_operator,1.0,1.0
4,✅,센터명 키워드 검색 - 강남,search_center_by_name,1.0,1.0
5,✅,서울 강남구 치매안심센터 운영기관 조회,get_operator_by_center,1.0,1.0
6,✅,센터 제공 프로그램 조회,get_programs_by_center,1.0,1.0
7,✅,시도 목록 조회,get_sido_list,1.0,1.0
8,✅,시군구 목록 조회 - 경기도,get_sigungu_list,1.0,1.0
9,✅,flexible - 의사 2명 이상 센터 (fallback),flexible_graph_search,1.0,1.0


---
## 2. VectorDB 성능 평가 (Hit Rate / MRR)

질의-정답 키워드 쌍으로 top-k 검색 결과에 키워드가 포함되면 hit로 판정한다.  
`SCORE_THRESHOLD=0.3`, `TOP_K=4` 기준.

In [10]:
from vector_db.vector_search_tool import search_dementia_guideline
print("VectorDB tool import 완료")

VectorDB tool import 완료


In [11]:
# 평가셋: (쿼리, 정답 키워드 목록)
VECTOR_EVAL_SET = [
    ("부모님이 자꾸 같은 말을 반복해요, 치매인가요?", ["반복", "기억", "치매"]),
    ("치매 조기검진은 어디서 어떻게 받나요?", ["검진", "치매안심센터", "보건소"]),
    ("알츠하이머랑 치매랑 뭐가 달라요?", ["알츠하이머", "치매"]),
    ("치매 예방하려면 어떻게 해야 하나요?", ["예방", "운동", "사회활동"]),
    ("경도인지장애가 뭔가요?", ["경도인지장애", "인지"]),
    ("치매 환자 가족으로서 어떻게 대해야 하나요?", ["가족", "보호자", "대응"]),
    ("치매 검사 비용이 얼마나 드나요?", ["비용", "무료", "검사"]),
    ("밤에 자꾸 나가려고 해요, 치매 증상인가요?", ["배회", "증상"]),
    ("치매 관련 법적 지원이 있나요?", ["법", "지원", "제도"]),
    ("파킨슨병과 치매의 관계가 뭔가요?", ["파킨슨", "치매", "인지"]),
]

In [12]:
vector_results = []
hit_count = 0
mrr_total = 0.0

for query, keywords in VECTOR_EVAL_SET:
    result = search_dementia_guideline.invoke({"query": query})
    
    # 결과를 청크별로 분리 (포맷: [참고자료 N] ...)
    chunks = result.split("[참고자료")
    
    found_rank = None
    for rank, chunk in enumerate(chunks[1:], 1):  # [0]은 빈 문자열
        if any(kw in chunk for kw in keywords):
            found_rank = rank
            break
    
    hit = found_rank is not None
    mrr = 1 / found_rank if found_rank else 0
    
    if hit:
        hit_count += 1
    mrr_total += mrr
    
    status = "✅" if hit else "❌"
    print(f"{status} {query[:40]}... | rank={found_rank} MRR={mrr:.4f}")
    vector_results.append({"query": query, "hit": hit, "rank": found_rank, "mrr": mrr})

n = len(VECTOR_EVAL_SET)
hit_rate = hit_count / n
mean_mrr = mrr_total / n
print(f"\nHit Rate: {hit_count}/{n} = {hit_rate:.4f}")
print(f"MRR:      {mean_mrr:.4f}")

✅ 부모님이 자꾸 같은 말을 반복해요, 치매인가요?... | rank=1 MRR=1.0000
✅ 치매 조기검진은 어디서 어떻게 받나요?... | rank=1 MRR=1.0000
✅ 알츠하이머랑 치매랑 뭐가 달라요?... | rank=1 MRR=1.0000
✅ 치매 예방하려면 어떻게 해야 하나요?... | rank=3 MRR=0.3333
✅ 경도인지장애가 뭔가요?... | rank=1 MRR=1.0000
✅ 치매 환자 가족으로서 어떻게 대해야 하나요?... | rank=1 MRR=1.0000
✅ 치매 검사 비용이 얼마나 드나요?... | rank=1 MRR=1.0000
✅ 밤에 자꾸 나가려고 해요, 치매 증상인가요?... | rank=1 MRR=1.0000
✅ 치매 관련 법적 지원이 있나요?... | rank=1 MRR=1.0000
✅ 파킨슨병과 치매의 관계가 뭔가요?... | rank=1 MRR=1.0000

Hit Rate: 10/10 = 1.0000
MRR:      0.9333


In [13]:
df_vector = pd.DataFrame(vector_results)
df_vector["status"] = df_vector["hit"].map({True: "✅", False: "❌"})
df_vector[["status", "query", "rank", "mrr"]]

,status,query,rank,mrr
0,✅,"부모님이 자꾸 같은 말을 반복해요, 치매인가요?",1,1.000000
1,✅,치매 조기검진은 어디서 어떻게 받나요?,1,1.000000
2,✅,알츠하이머랑 치매랑 뭐가 달라요?,1,1.000000
3,✅,치매 예방하려면 어떻게 해야 하나요?,3,0.333333
4,✅,경도인지장애가 뭔가요?,1,1.000000
5,✅,치매 환자 가족으로서 어떻게 대해야 하나요?,1,1.000000
6,✅,치매 검사 비용이 얼마나 드나요?,1,1.000000
7,✅,"밤에 자꾸 나가려고 해요, 치매 증상인가요?",1,1.000000
8,✅,치매 관련 법적 지원이 있나요?,1,1.000000
9,✅,파킨슨병과 치매의 관계가 뭔가요?,1,1.000000


---
## 3. RAG 모델 성능 평가 (RAGAS)

### 흐름
1. Qdrant에서 전체 청크 로드 → LangChain `Document` 변환
2. `TestsetGenerator`로 테스트셋 20개 자동 생성 (simple / reasoning / multi_context 혼합)
3. `get_structured_answer`로 각 질문에 대한 응답 생성
4. RAGAS 4개 지표 측정

In [2]:
# 노트북 첫 셀에 이거 먼저 실행
import sys
from unittest.mock import MagicMock

# vertexai mock으로 우회
mock = MagicMock()
sys.modules['langchain_community.chat_models.vertexai'] = mock
sys.modules['langchain_community.llms.vertexai'] = mock

# 이후 정상 import
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import EvaluationDataset, evaluate
print("ok")

ok


In [7]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import EvaluationDataset, evaluate
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, Faithfulness, AnswerRelevancy
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 래핑
generator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))
eval_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

# 평가 지표
metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings),
]

C:\Users\Playdata\AppData\Local\Temp\ipykernel_48020\2609202321.py:5: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, Faithfulness, AnswerRelevancy
C:\Users\Playdata\AppData\Local\Temp\ipykernel_48020\2609202321.py:5: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, Faithfulness, AnswerRelevancy
C:\Users\Playdata\AppData\Local\Temp\ipykernel_48020\2609202321.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be re

In [8]:
from qdrant_client import QdrantClient
from langchain_core.documents import Document
import os

qdrant = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

all_points = []
offset = None
while True:
    batch, offset = qdrant.scroll(
        collection_name="dementia_guideline",
        limit=100,
        offset=offset,
        with_payload=True,
        with_vectors=False
    )
    all_points.extend(batch)
    if offset is None:
        break

docs = [
    Document(
        page_content=p.payload["text"],
        metadata={
            "title": p.payload.get("title", ""),
            "source": p.payload.get("source_url", ""),
        }
    )
    for p in all_points
]
print(f"청크 로드 완료: {len(docs)}개")

청크 로드 완료: 93개


In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

testset = generator.generate_with_langchain_docs(
    docs,
    testset_size=15,
)

eval_df = testset.to_pandas()
print(f"테스트셋 생성 완료: {len(eval_df)}개")
eval_df[["user_input", "reference", "synthesizer_name"]].head(5)

In [ ]:
import sys
import time
sys.path.insert(0, os.path.abspath(".."))

from modules.agent import get_structured_answer
from vector_db.vector_search_tool import search_dementia_guideline

responses = []
contexts_list = []

n_total = len(eval_df)
for i, (_, row) in enumerate(eval_df.iterrows(), 1):
    question = row["user_input"]
    t0 = time.time()

    # 에이전트 응답
    try:
        structured = get_structured_answer(question)
        answer = structured["content"]["text"] if structured["type"] == "reply" else structured["content"]["question"]
    except Exception as e:
        answer = f"오류: {e}"

    # VectorDB 컨텍스트
    vector_result = search_dementia_guideline.invoke({"query": question})
    contexts = [c.strip() for c in vector_result.split("[참고자료") if c.strip()]
    contexts = contexts if contexts else [vector_result]

    responses.append(answer)
    contexts_list.append(contexts)

    elapsed = time.time() - t0
    print(f"[{i}/{n_total}] ({elapsed:.1f}초) {question[:50]}...")

eval_df["response"] = responses
eval_df["retrieved_contexts"] = contexts_list
print("응답 생성 완료")


In [1]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, Faithfulness, AnswerRelevancy

eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings),
]

eval_result = evaluate(dataset=eval_dataset, metrics=metrics)
print(eval_result)

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
# 질문별 상세 결과
df_ragas = ragas_result.to_pandas()
df_ragas[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]]

---
## 4. 종합 요약

In [ ]:
print("=" * 50)
print("[ 성능 평가 종합 결과 ]")
print("=" * 50)
print(f"\n[GraphDB]")
print(f"  평균 Precision : {avg_p:.4f}")
print(f"  평균 Recall    : {avg_r:.4f}")
print(f"\n[VectorDB]")
print(f"  Hit Rate : {hit_rate:.4f}  ({hit_count}/{n})")
print(f"  MRR      : {mean_mrr:.4f}")
print(f"\n[RAGAS]")
print(f"  Faithfulness      : {ragas_result['faithfulness']:.4f}")
print(f"  Answer Relevancy  : {ragas_result['answer_relevancy']:.4f}")
print(f"  Context Precision : {ragas_result['context_precision']:.4f}")
print(f"  Context Recall    : {ragas_result['context_recall']:.4f}")
print("=" * 50)